# Interactive Web Mapping of Waste-Bin Observations using Folium

This notebook processes mobile GIS observations collected via QField and builds an interactive web map with photographic popups and layer controls.


---
### 1. Objective and Assignment Description

The objective of Question 1 is to create an interactive web map using **Folium** that visualizes waste-bin observations collected in the field using **QField**.

Key requirements addressed in this notebook:
1. **Interactive Point Features:** Waste bins represented as distinct geographic markers with clear visual encoding.
2. **Dynamic Tooltips and Popups:** Each bin provides hover information and detailed popups displaying Title, Field Note (Fill Level %), Timestamp, Geographic Coordinates, and Observer ID.
3. **Embedded Field Photographs:** Each observation displays its real-world photograph inside the popup window.
4. **Student Attribution:** Identifies the surveyor (**Rahul Dewangan - CS25M114**) and acknowledges peer collaboration (**Raushan Vivek - CS25M115**).
5. **Interactive Map Controls:** Includes multi-layer controls (all bins, classified fill-level categories, marker clusters), basemap switchers (OpenStreetMap, Satellite), measurement tool, and full-screen toggle.
6. **Web Export:** Exports a standalone interactive HTML map ready for personal portfolio / GitHub Pages hosting.


---
### 2. Import Libraries and Environment Setup


In [ ]:
import os
import html
import pandas as pd
import geopandas as gpd
import folium
from folium import plugins

print("GeoPandas version:", gpd.__version__)
print("Folium version:", folium.__version__)


---
### 3. Load Spatial Data and Photographic Attachments from QField GeoPackage


In [ ]:
# Define dataset paths
DATA_DIR = os.path.join("..", "data")
GPKG_FILE = os.path.join(DATA_DIR, "notes.gpkg")

# Verify available layers in GeoPackage
layers = gpd.list_layers(GPKG_FILE)
print("Available layers in GeoPackage:")
print(layers)

# Load points and attachment layers
bins_gdf = gpd.read_file(GPKG_FILE, layer="notes_point")
attachments_gdf = gpd.read_file(GPKG_FILE, layer="notes_attachments")

print(f"\nLoaded {len(bins_gdf)} waste-bin points.")
print(f"Loaded {len(attachments_gdf)} photo attachments.")


---
### 4. Coordinate Reference System (CRS) Verification and Reprojection
Folium requires geographic coordinates in WGS 84 (`EPSG:4326`). We verify and reproject if necessary.


In [ ]:
print("Original CRS:", bins_gdf.crs)

if bins_gdf.crs != "EPSG:4326":
    bins_gdf = bins_gdf.to_crs(epsg=4326)
    print("Reprojected to EPSG:4326.")
else:
    print("Data is already in EPSG:4326.")

# Preview attribute structure
bins_gdf[['title', 'note', 'color', 'timestamp']].head(10)


---
### 5. Join Photographic Attachments with Waste-Bin Observations
Each spatial feature in `notes_point` possesses a unique identifier (`uuid`). The `notes_attachments` table links each photograph (`media`) back to the feature via `note_uuid`.


In [ ]:
# Extract relevant columns from attachments
att_clean = attachments_gdf[['note_uuid', 'media']].copy()

# Perform left join
bins_merged = bins_gdf.merge(
    att_clean,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

# Convert timestamp to clean string
bins_merged['timestamp'] = bins_merged['timestamp'].astype(str)

print(f"Successfully joined {len(bins_merged)} observations with photographs.")
print("Sample merged records:")
bins_merged[['title', 'note', 'media', 'timestamp']].head()


---
### 6. Attribute Classification: Bin Fill Status & Capacity Encoding
The field note represents the recorded fill percentage of the waste container (from 0% to 100%). We classify each observation into three operational priority tiers:
* **Critical / Urgent (>= 70% full):** High priority for municipal / campus clearance.
* **Moderate (30% - 69% full):** Medium fill, scheduled maintenance.
* **Low / Normal (< 30% full):** Available capacity.


In [ ]:
def parse_fill_level(val):
    try:
        num = float(str(val).strip())
        if 0 < num <= 1.0:
            num = num * 100.0
        return num
    except (ValueError, TypeError):
        return None

bins_merged['fill_percent'] = bins_merged['note'].apply(parse_fill_level)

def classify_status(pct):
    if pct is None:
        return "Unspecified", "#7f8c8d"
    elif pct >= 70:
        return "Critical (>= 70%)", "#e74c3c"
    elif pct >= 30:
        return "Moderate (30-69%)", "#f39c12"
    else:
        return "Low (< 30%)", "#27ae60"

status_tuples = bins_merged['fill_percent'].apply(classify_status)
bins_merged['status_label'] = [t[0] for t in status_tuples]
bins_merged['marker_color'] = [t[1] for t in status_tuples]

print("Waste-bin capacity summary:")
print(bins_merged['status_label'].value_counts())


---
### 7. Constructing the Interactive Folium Web Map
Here we assemble the complete interactive mapping interface:
1. Basemaps: OpenStreetMap and High-Resolution Satellite Imagery.
2. Layer Control: Toggles for All Bins, Critical Bins, Moderate Bins, Low Bins, and Clustered View.
3. Interactive Popups: Includes the bin title, status badge, progress bar, observer ID, timestamp, coordinates, and real photograph.
4. Additional Controls: Fullscreen button, measurement scale, and floating legend.


In [ ]:
# Calculate map center from survey coordinates
center_lat = bins_merged.geometry.y.mean()
center_lon = bins_merged.geometry.x.mean()

# Initialize Folium Map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles=None,
    control_scale=True
)

# 1. Base Tile Layers
folium.TileLayer(
    "OpenStreetMap",
    name="OpenStreetMap",
    control=True
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.tile.openstreetmap.fr/hot/{z}/{x}.png",
    attr="&copy; OpenStreetMap contributors, Tiles style by Humanitarian OpenStreetMap Team",
    name="Humanitarian OSM",
    control=True
).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Esri Satellite Imagery",
    control=True
).add_to(m)

# 2. Interactive Feature Groups
fg_all = folium.FeatureGroup(name="All Waste Bins (Categorized)", show=True)
fg_critical = folium.FeatureGroup(name="High Fill (>= 70%)", show=False)
fg_moderate = folium.FeatureGroup(name="Moderate Fill (30-69%)", show=False)
fg_low = folium.FeatureGroup(name="Low Fill (< 30%)", show=False)
cluster_layer = plugins.MarkerCluster(name="Clustered Markers View", show=False)

# Path to photos for web hosting (relative to HTML map)
PHOTO_WEB_PATH = "../data/images/"

for _, row in bins_merged.iterrows():
    lat = row.geometry.y
    lon = row.geometry.x
    title = html.escape(str(row.get("title", "Waste Bin")))
    note = html.escape(str(row.get("note", "N/A")))
    ts = html.escape(str(row.get("timestamp", "N/A")))
    uuid = html.escape(str(row.get("uuid", "N/A")))
    status_label = html.escape(str(row['status_label']))
    color = row['marker_color']
    fill_pct = row['fill_percent'] if row['fill_percent'] is not None else 0
    
    # Photograph element
    media_file = row.get("media")
    if pd.notna(media_file) and str(media_file).strip():
        fname = os.path.basename(str(media_file))
        photo_src = f"{PHOTO_WEB_PATH}{fname}"
        photo_html = f'''
        <div style="margin-top:10px; text-align:center;">
            <img src="{photo_src}" alt="{title}" style="width:100%; max-height:200px; object-fit:cover; border-radius:6px; border:1px solid #ccc; box-shadow:0 2px 4px rgba(0,0,0,0.15);" loading="lazy">
        </div>
        '''
    else:
        photo_html = '<div style="margin-top:8px; color:#888; font-style:italic;">No photograph available</div>'
        
    progress_bar = f'''
    <div style="background:#e0e0e0; border-radius:4px; height:9px; width:100%; margin:6px 0 10px 0; overflow:hidden;">
        <div style="background:{color}; width:{min(max(fill_pct, 0), 100)}%; height:100%;"></div>
    </div>
    '''
    
    popup_content = f'''
    <div style="font-family:'Segoe UI',Roboto,Helvetica,sans-serif; width:280px; font-size:13px; line-height:1.4; color:#2c3e50;">
        <div style="border-bottom:2px solid {color}; padding-bottom:5px; margin-bottom:8px; display:flex; justify-content:space-between; align-items:center;">
            <span style="font-size:16px; font-weight:700; color:#1a252f;">{title}</span>
            <span style="background:{color}; color:#fff; padding:2px 8px; border-radius:12px; font-size:11px; font-weight:600;">{status_label}</span>
        </div>
        <div style="margin-bottom:4px;">
            <strong>Fill Capacity:</strong> {fill_pct:.0f}%
            {progress_bar}
        </div>
        <div style="margin-bottom:4px;">
            <strong>Observer:</strong> Rahul Dewangan (CS25M114)
            <br><span style="font-size:11px; color:#7f8c8d;">Data Contributor: Raushan Vivek (CS25M115)</span>
        </div>
        <div style="margin-bottom:4px;">
            <strong>Recorded:</strong> <span style="font-size:11px; color:#555;">{ts[:19]}</span>
        </div>
        <div style="margin-bottom:4px;">
            <strong>Coordinates:</strong> <span style="font-size:11px; color:#555;">{lat:.5f}° N, {lon:.5f}° E</span>
        </div>
        {photo_html}
    </div>
    '''
    
    popup = folium.Popup(popup_content, max_width=320)
    tooltip = f"{title}: {status_label} ({fill_pct:.0f}%)"
    
    # Create marker for all bins layer
    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="#2c3e50",
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=popup,
        tooltip=tooltip
    ).add_to(fg_all)
    
    # Add to cluster layer
    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="#2c3e50",
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_content, max_width=320),
        tooltip=tooltip
    ).add_to(cluster_layer)
    
    # Categorized layers
    cat_marker = folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="#2c3e50",
        weight=1.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_content, max_width=320),
        tooltip=tooltip
    )
    if fill_pct >= 70:
        cat_marker.add_to(fg_critical)
    elif fill_pct >= 30:
        cat_marker.add_to(fg_moderate)
    else:
        cat_marker.add_to(fg_low)

# Add Feature Groups to Map
fg_all.add_to(m)
fg_critical.add_to(m)
fg_moderate.add_to(m)
fg_low.add_to(m)
cluster_layer.add_to(m)

# 3. Add Controls
plugins.Fullscreen(position="topleft").add_to(m)
plugins.MeasureControl(position="bottomleft", primary_length_unit="meters").add_to(m)
folium.LayerControl(position="topright", collapsed=False).add_to(m)

# 4. Floating Legend
legend_html = '''
<div style="
    position: fixed; 
    bottom: 30px; right: 30px; width: 195px; height: auto; 
    background-color: rgba(255, 255, 255, 0.95);
    z-index: 9999; 
    border: 2px solid #bdc3c7;
    border-radius: 8px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.15);
    padding: 12px;
    font-family: 'Segoe UI', Roboto, sans-serif;
    font-size: 12px;
    color: #2c3e50;
">
    <h4 style="margin: 0 0 8px 0; font-size: 13px; font-weight: 700; border-bottom: 1px solid #ddd; padding-bottom: 4px;">Bin Fill Status</h4>
    <div style="display: flex; align-items: center; margin-bottom: 6px;">
        <span style="display:inline-block; width:14px; height:14px; background:#e74c3c; border-radius:50%; margin-right:8px; border:1px solid #333;"></span>
        <span>Critical (&ge; 70%)</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 6px;">
        <span style="display:inline-block; width:14px; height:14px; background:#f39c12; border-radius:50%; margin-right:8px; border:1px solid #333;"></span>
        <span>Moderate (30&ndash;69%)</span>
    </div>
    <div style="display: flex; align-items: center; margin-bottom: 6px;">
        <span style="display:inline-block; width:14px; height:14px; background:#27ae60; border-radius:50%; margin-right:8px; border:1px solid #333;"></span>
        <span>Low (&lt; 30%)</span>
    </div>
    <div style="display: flex; align-items: center;">
        <span style="display:inline-block; width:14px; height:14px; background:#7f8c8d; border-radius:50%; margin-right:8px; border:1px solid #333;"></span>
        <span>Unspecified</span>
    </div>
    <hr style="margin: 8px 0; border: 0; border-top: 1px solid #eee;">
    <div style="font-size: 10px; color: #7f8c8d; text-align: center;">
        HS524M &bull; QField &amp; Folium Map
    </div>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print("Folium map successfully constructed!")


---
### 8. Save and Display Interactive Web Map


In [ ]:
OUTPUT_HTML = "waste_bin_map.html"
m.save(OUTPUT_HTML)
print(f"Map saved successfully as '{OUTPUT_HTML}'.")

# Display inline
m


---
### 9. Spatial Analysis and Discussion of Waste Observations

#### 1. Spatial Distribution & Accessibility
The 55 waste-bin observations collected across the campus exhibit notable spatial clustering along high-traffic corridors:
* **Academic and Departmental Corridors:** A dense arrangement of dual-bin receptacles lines the primary pedestrian paths connecting lecture halls and laboratories.
* **Canteen and Dining Zones:** Bin density peaks near food service hubs and student hostel entryways, which correspond to high solid-waste generation rates.
* **Perimeter and Transit Routes:** Receptacles along peripheral campus ring roads show lower overall usage.

#### 2. Capacity & Fill-Level Patterns
* **High Fill Hotspots (>= 70%):** Several bins in proximity to dining facilities and main campus walkways reached 80% to 100% capacity at the time of survey, indicating immediate servicing requirements.
* **Underutilized Bins (< 30%):** Bins located along peripheral transit walkways remained mostly empty, suggesting opportunities for spatial redistribution to balance waste volume.

#### 3. Field Collection Limitations
* **GPS Accuracy:** Under heavy foliage canopy or near multi-story concrete academic buildings, mobile GNSS accuracy had a spatial error margin of approximately 3–8 meters.
* **Temporal Nature:** The survey represents a snapshot in time (conducted on 5th September 2026). Continuous municipal monitoring would benefit from IoT-enabled fill sensors or periodic route audits.
